In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
import chromadb

load_dotenv()

LM_MINI_MODEL = os.getenv("MINIML_MODEL_PATH")
MISTRAL_API = os.getenv("MISTRAL_API", "http://localhost:1234/v1")

llm = ChatOpenAI(
        temperature=0,
        model_name="local-model",
        base_url=MISTRAL_API,
        api_key="api-key"
    )


In [3]:

path = "../../resources/customer-100.csv"
# file_path = Path(path)
data = pd.read_csv(path)
data.head()

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/


In [4]:
loader = CSVLoader(file_path=path)
docs = loader.load_and_split()
docs

[Document(metadata={'source': '../../resources/customer-100.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
 Document(metadata={'source': '../../resources/customer-100.csv', 'row': 1}, page_content='Index: 2\nCustomer Id: 1Ef7b82A4CAAD10\nFirst Name: Preston\nLast Name: Lozano\nCompany: Vega-Gentry\nCity: East Jimmychester\nCountry: Djibouti\nPhone 1: 5153435776\nPhone 2: 686-620-1820x944\nEmail: vmata@colon.com\nSubscription Date: 2021-04-23\nWebsite: http://www.hobbs.com/'),
 Document(metadata={'source': '../../resources/customer-100.csv', 'row': 2}, page_content='Index: 3\nCustomer Id: 6F94879bDAfE5a6\nFirst Name: Roy\nLast Name: Berry\nCompany: Murillo-Perry\nCity: Isabelborough\nCountry: Antigua and Barbuda\n

In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [6]:
embeddings = HuggingFaceEmbeddings(model_name=LM_MINI_MODEL)
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,    
)

C:\Users\scientist\AppData\Local\Temp\ipykernel_18804\190986226.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=LM_MINI_MODEL)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17542.26it/s]


In [7]:
client = chromadb.Client(chromadb.config.Settings(persist_directory="../../resources/databases",
                         chroma_db_impl="duckdb+parquet"))
vector_chroma_store = Chroma.from_docuemnts(
    documents=docs,
    embedding=embeddings,
    persist_directory="../../resources/databases",
    client=client
)

# Persist
vector_store.persist()

ValueError: [91mYou are using a deprecated configuration of Chroma.

[94mIf you do not have data you wish to migrate, you only need to change how you construct
your Chroma client. Please see the "New Clients" section of https://docs.trychroma.com/deployment/migration.
________________________________________________________________________________________________

If you do have data you wish to migrate, we have a migration tool you can use in order to
migrate your data to the new Chroma architecture.
Please `pip install chroma-migrate` and run `chroma-migrate` to migrate your data and then
change how you construct your Chroma client.

See https://docs.trychroma.com/deployment/migration for more information or join our discord at https://discord.gg/MMeYNTmh3x for help![0m

In [14]:
import chromadb
from langchain_chroma import Chroma  # If using LangChain

# New way - simpler client initialization
client = chromadb.PersistentClient(
    path="../../resources/databases"  # persist_directory becomes path
)

# Create the vector store
vector_chroma_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
    collection_name="customer_data"  # Specify collection name
)

# Persist is automatic with PersistentClient, but if needed:
# vector_chroma_store.persist()  # Often not required anymore